In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

In [ ]:
data_dir = (Path.cwd() if (Path.cwd() / "mcs_ds_edited_iter_shuffled.csv").exists() else Path.cwd() / "homework_1")
[f.name for f in data_dir.iterdir()]

In [ ]:
df = pd.read_csv(data_dir / "mcs_ds_edited_iter_shuffled.csv")

display(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing values:")
print(df.isna().sum())

In [ ]:
feature_cols = df.columns[:4].tolist()
target_col = "ale"  

print("Features:", feature_cols)
print("Target:", target_col)

data = df[feature_cols + [target_col]].copy()
data = data.apply(pd.to_numeric, errors="raise")

print("Rows with missing values:", data.isna().any(axis=1).sum())
data = data.dropna()

X = data[feature_cols]
y = data[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
models = {
    "Linear Regression": LinearRegression(),

    "Neural Network": make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(32, 16),
            activation="relu",
            solver="lbfgs",
            max_iter=5000,
            random_state=42
        )
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
}

results = []
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    residuals = y_test.to_numpy() - predictions

    statistic, p_value = stats.normaltest(residuals)

    results.append({
        "Model": name,
        "Statistic": statistic,
        "p-value": p_value,
        "Reject normality (5%)": p_value < 0.05,
    })

    stats.probplot(residuals, dist="norm", plot=ax)
    ax.set_title(name)

fig.tight_layout()
fig.savefig(data_dir / "HW1_4_qqplots.png", dpi=300)
plt.show()

results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv(data_dir / "HW1_4_results.csv", index=False)

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, model) in zip(axes, models.items()):
    predictions = model.predict(X_test)
    residuals = y_test.to_numpy() - predictions

    ax.hist(
        residuals,
        bins="auto",
        density=True,
        alpha=0.65,
        color="steelblue",
        edgecolor="white",
        label="Residuals"
    )
    
    mu, sigma = stats.norm.fit(residuals)
    if sigma > 0:
        x = np.linspace(
            min(residuals.min(), mu - 3 * sigma),
            max(residuals.max(), mu + 3 * sigma),
            300
        )
        ax.plot(
            x, stats.norm.pdf(x, loc=mu, scale=sigma),
            color="red", linewidth=2,
            label="Fitted normal"
        )

    ax.axvline(0, color="black", linestyle="--", alpha=0.5)
    ax.set_title(name)
    ax.set_xlabel("Residual (observed - predicted)")
    ax.set_ylabel("Density")
    ax.legend()

fig.tight_layout()
fig.savefig(data_dir / "HW1_4_residual_distributions.png", dpi=300)
plt.show()
